# Harness self-test

The widget below is fed by a WebSocket server that runs in a background thread of this kernel on 127.0.0.1. It keeps drawing while the next cell runs a loop, because its data does not travel through Jupyter's message channel.

In [ ]:
import json
import os
import time

import numpy as np
from mock_stream import MockCanvas, MockStreamServer

server = MockStreamServer(rate=30)
widget = MockCanvas(src=server.url)
widget

In [ ]:
# Keep the kernel busy, as a training loop would. The loop ends after
# MOCK_LOOP_SECONDS (default 8) or when a file named stop-loop appears.
seconds = float(os.environ.get("MOCK_LOOP_SECONDS", "8"))
a = np.random.default_rng(0).standard_normal((256, 256)).astype(np.float32)
steps, start = 0, time.monotonic()
while time.monotonic() - start < seconds and not os.path.exists("stop-loop"):
    a = np.tanh(a @ a.T / 256.0)
    sum(range(20000))
    steps += 1
print(f"loop ran {steps} steps")

In [ ]:
# Closed first: the server waits for its connections to end, so every
# message a widget sent before then is in the report.
server.close()
report = server.write_report("mock_stream_report.json")
print(report["frames_generated"], "frames;", len(report["connections"]), "connections")